# PaddleOCR-VL FastAPI Server via Ngrok

This notebook turns your Google Colab instance into a backend API server. It exposes an endpoint that accepts an image upload and dynamic hyperparameters, processes it using the 0.9B VLM, and returns the bounding boxes and reconstructed Markdown.

## 1. Install Dependencies
**NOTE:** If you see any errors during this installation, ignore them. Wait for the cell to finish, then go to **Runtime > Restart session** and proceed to the next cell. The errors are false alarms due to Colab's pre-loaded C-binaries!

In [1]:
# Install 'uv' for extremely fast installations
!pip install -q uv

!uv pip install --system paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/
!uv pip install --system "paddleocr[all]" "paddleocr[doc-parser]" imgaug
!uv pip install --system --reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!uv pip install --system -U Pillow
!uv pip install --system fastapi uvicorn python-multipart pyngrok nest-asyncio
!wget -q -O simfang.ttf https://raw.githubusercontent.com/PaddlePaddle/PaddleOCR/release/2.7/doc/fonts/simfang.ttf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 89.8 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 25 packages in 8.91s
Prepared 13 packages in 2m 14s
Uninstalled 1 package in 2ms
Installed 13 packages in 125ms
 + nvidia-cublas-cu11==11.11.3.6
 + nvidia-cuda-cupti-cu11==11.8.87
 + nvidia-cuda-nvrtc-cu11==11.8.89
 + nvidia-cuda-runtime-cu11==11.8.89
 + nvidia-cudnn-cu11==8.9.6.50
 + nvidia-cufft-cu11==10.9.0.58
 + nvidia-curand-cu11==10.3.0.86
 + nvidia-cusolver-cu11==11.4.1.48
 + nvidia-cusparse-cu11==11.7.5.86
 + nvidia-nccl-cu11==2.19.3
 + nvidia-nvtx-cu11==11.8.86
 - opt-einsum==3.4.0
 + opt-einsum==3.3.0
 + paddlepaddle-gpu==3.2.0
Using Python 3.12.13 environment at: /usr
Resolved 130 packages in 1.42s
Prepared 39 packages in 2.56s
Uninstalled 6 packages in 45ms
Installed 39 packages in 155ms
 + aistudio-sdk==0.3.9
 + bce-python-sdk==0.9.76
 + colorlog==6.12.0
 + crc32c==2.8
 + cssselect==1.5.0
 + cssutils==2.15.0
 + dataclasses-json==0.6.7
 + encutil

## 2. Configure ngrok Tunnel

1. Sign up for a free account at [ngrok.com](https://dashboard.ngrok.com/signup)
2. Copy your auth token from [dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)
3. Paste it below and run the cell

In [1]:
NGROK_AUTH_TOKEN = "YOUR_NGROK_TOKEN"  # @param {type:"string"}

if not NGROK_AUTH_TOKEN:
    raise ValueError(
        "Please set your ngrok auth token above.\n"
        "Get one free at: https://dashboard.ngrok.com/get-started/your-authtoken"
    )

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print("ngrok configured.")

ngrok configured.


## 3. Initialize the VLM Globally

In [2]:
import os
import gc
import warnings
import logging
import torch

# Aggressively suppress warnings to keep server logs clean
warnings.filterwarnings('ignore')
logging.getLogger('ppocr').setLevel(logging.ERROR)
logging.getLogger('paddlex').setLevel(logging.ERROR)
os.environ['FLAGS_enable_pir_api'] = '0'

from paddleocr import PaddleOCRVL

print("Loading the 0.9B VLM into GPU Memory. This will take a moment...")
# Initialize globally so the server doesn't reload it per-request
pipeline = PaddleOCRVL()
print("Model Successfully Loaded!")

Loading the 0.9B VLM into GPU Memory. This will take a moment...


Creating model: ('PP-DocLayoutV3', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-DocLayoutV3), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-DocLayoutV3`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Creating model: ('PaddleOCR-VL-1.6-0.9B', None, None)
Using official model (PaddleOCR-VL-1.6), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PaddleOCR-VL-1.6`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]

Bucketed engine_config has no entry for resolved engine 'paddle_dynamic'; using an empty config for that engine.
Loading configuration file /root/.paddlex/official_models/PaddleOCR-VL-1.6/config.json
Loading weights file /root/.paddlex/official_models/PaddleOCR-VL-1.6/model.safetensors
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_he

Model Successfully Loaded!


## 4. Define and Start the API Server

## API Specification
**Endpoint:** `POST /api/ocr`  
**Content-Type:** `multipart/form-data`

### Request Parameters
| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| `file` | `file` | **Required** | The image file (jpg/png) |
| `temperature` | `float` | `0.0` | Controls randomness in VLM generation. 0.0 is deterministic. |
| `top_p` | `float` | `1.0` | Nucleus sampling probability. |
| `repetition_penalty` | `float` | `1.0` | Penalizes repeated tokens. |
| `layout_threshold` | `float` | `0.5` | Confidence threshold for the layout detection model. |
| `layout_nms` | `bool` | `true` | Whether to apply Non-Maximum Suppression to bounding boxes. |
| `max_new_tokens` | `int` | `2048` | Maximum output tokens generated by the VLM. |
| `min_pixels` | `int` | `null` | Minimum image resolution in pixels. |
| `max_pixels` | `int` | `null` | Maximum image resolution in pixels. |
| `use_doc_orientation_classify`| `bool` | `false` | Enable automatic rotation correction. |
| `use_doc_unwarping` | `bool` | `false` | Enable de-warping of curved text. |
| `use_layout_detection` | `bool` | `true` | Enable structural parsing of the document. |
| `use_chart_recognition` | `bool` | `false` | Enable chart parsing. |
| `use_seal_recognition` | `bool` | `false` | Enable stamp/seal recognition. |
| `use_ocr_for_image_block` | `bool` | `false` | Force traditional OCR on image layout blocks. |
| `format_block_content` | `bool` | `true` | Output formatted Markdown. |
| `merge_layout_blocks` | `bool` | `true` | Merge adjacent layout blocks. |

### Response JSON
```json
{
  "success": true,
  "markdown": "# Header\n\nExtracted text...",
  "boxes": [{"label": "text", "coordinate": [10, 10, 100, 50]}],
  "parameters_used": { ... }
}
```

In [3]:
import io
import traceback
from typing import Optional
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware
import nest_asyncio
import uvicorn

app = FastAPI(title="PaddleOCR-VL API")

# Allow all CORS (useful if you ever hit it from a web frontend instead of just Android)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.post("/api/ocr")
async def perform_ocr(
    file: UploadFile = File(...),
    temperature: float = Form(0.0),
    top_p: float = Form(1.0),
    repetition_penalty: float = Form(1.0),
    layout_threshold: float = Form(0.5),
    layout_nms: bool = Form(True),
    max_new_tokens: int = Form(2048),
    min_pixels: Optional[int] = Form(None),
    max_pixels: Optional[int] = Form(None),
    use_doc_orientation_classify: bool = Form(False),
    use_doc_unwarping: bool = Form(False),
    use_layout_detection: bool = Form(True),
    use_chart_recognition: bool = Form(False),
    use_seal_recognition: bool = Form(False),
    use_ocr_for_image_block: bool = Form(False),
    format_block_content: bool = Form(True),
    merge_layout_blocks: bool = Form(True)
):
    """
    Accepts a multipart file upload from the Android App and runs the VLM pipeline.
    """
    print(f"Received request for {file.filename} | Temp: {temperature} | Thresh: {layout_threshold}")
    try:
        # 1. Save uploaded image to a temporary file
        img_bytes = await file.read()
        img_path = "/tmp/upload_api_img.jpg"
        with open(img_path, "wb") as f:
            f.write(img_bytes)

        # 2. Run the VLM pipeline synchronously (using use_queues=False to prevent OOM)
        output = list(pipeline.predict(
            [img_path],
            use_queues=False,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            layout_threshold=layout_threshold,
            layout_nms=layout_nms,
            max_new_tokens=max_new_tokens,
            min_pixels=min_pixels,
            max_pixels=max_pixels,
            use_doc_orientation_classify=use_doc_orientation_classify,
            use_doc_unwarping=use_doc_unwarping,
            use_layout_detection=use_layout_detection,
            use_chart_recognition=use_chart_recognition,
            use_seal_recognition=use_seal_recognition,
            use_ocr_for_image_block=use_ocr_for_image_block,
            format_block_content=format_block_content,
            merge_layout_blocks=merge_layout_blocks
        ))[0]

        # 3. Parse the results from the PaddleX output dictionary
        res_dict = output.res if hasattr(output, 'res') else output

        # 4. Save to Markdown and read it back (Easiest way to get the final compiled VLM text)
        md_dir = "/tmp/md_out"
        os.makedirs(md_dir, exist_ok=True)
        markdown_text = ""

        if hasattr(output, 'save_to_markdown'):
            output.save_to_markdown(md_dir)
            # It saves as the original filename minus extension, plus '.md'
            md_path = os.path.join(md_dir, "upload_api_img.md")
            if os.path.exists(md_path):
                with open(md_path, "r", encoding="utf-8") as f:
                    markdown_text = f.read()

        # 5. Extract bounding boxes so the Android app can draw UI overlays
        boxes = []
        if 'layout_det_res' in res_dict and 'boxes' in res_dict['layout_det_res']:
            for box in res_dict['layout_det_res']['boxes']:
                clean_box = {}
                for k, v in box.items():
                    if hasattr(v, 'tolist'):
                        clean_box[k] = v.tolist()
                    else:
                        clean_box[k] = v
                boxes.append(clean_box)
        gc.collect()

        return JSONResponse(content={
            "success": True,
            "markdown": markdown_text,
            "boxes": boxes,
            "parameters_used": {
                "temperature": temperature,
                "top_p": top_p,
                "repetition_penalty": repetition_penalty,
                "layout_threshold": layout_threshold,
                "layout_nms": layout_nms,
                "max_new_tokens": max_new_tokens,
                "min_pixels": min_pixels,
                "max_pixels": max_pixels,
                "use_doc_orientation_classify": use_doc_orientation_classify,
                "use_doc_unwarping": use_doc_unwarping,
                "use_layout_detection": use_layout_detection,
                "use_chart_recognition": use_chart_recognition,
                "use_seal_recognition": use_seal_recognition,
                "use_ocr_for_image_block": use_ocr_for_image_block,
                "format_block_content": format_block_content,
                "merge_layout_blocks": merge_layout_blocks
            }
        })
    except Exception as e:
        print(f"API Error: {e}")
        return JSONResponse(status_code=500, content={
            "success": False,
            "error": str(e),
            "trace": traceback.format_exc()
        })



In [4]:
try:
    # Disconnect any existing tunnels on this port
    tunnels = ngrok.get_tunnels()
    for tunnel in tunnels:
        if "8000" in tunnel.config['addr']:
            ngrok.disconnect(tunnel.public_url)
except Exception as e:
    pass

# Open a fresh ngrok tunnel to the FastAPI port (8000)
public_url = ngrok.connect(8000).public_url
print("\n" + "=" * 70)
print("🚀 NGROK TUNNEL OPENED SUCCESSFULLY!")
print(f"👉 COPY THIS URL: {public_url}")
print(f"👉 API Endpoint:  {public_url}/api/ocr")
print("=" * 70 + "\n")
print("Now run the final cell below to start the server! (It will block this tab)")



🚀 NGROK TUNNEL OPENED SUCCESSFULLY!
👉 COPY THIS URL: https://facecloth-askew-esquire.ngrok-free.dev
👉 API Endpoint:  https://facecloth-askew-esquire.ngrok-free.dev/api/ocr

Now run the final cell below to start the server! (It will block this tab)


In [ ]:
# Start the server natively in Colab
import uvicorn
print("🚀 PADDLEOCR API SERVER IS STARTING...")
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()


🚀 PADDLEOCR API SERVER IS STARTING...


INFO:     Started server process [6411]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Received request for Screenshot_20260813-110102_CPU-Z.jpg | Temp: 0.0 | Thresh: 0.5
INFO:     2a09:bac5:3c74:a8c::10d:25:0 - "POST /api/ocr HTTP/1.1" 200 OK
Received request for Screenshot_20260815-153527_Reddit.jpg | Temp: 0.0 | Thresh: 0.5
INFO:     2a09:bac5:3c74:a8c::10d:25:0 - "POST /api/ocr HTTP/1.1" 200 OK
Received request for Screenshot_20260815-153519_Reddit.jpg | Temp: 0.0 | Thresh: 0.5
INFO:     2a09:bac5:3c74:a8c::10d:25:0 - "POST /api/ocr HTTP/1.1" 200 OK
Received request for Screenshot_20260815-153437_WhatsApp.jpg | Temp: 0.0 | Thresh: 0.5
INFO:     2a09:bac5:3c74:a8c::10d:25:0 - "POST /api/ocr HTTP/1.1" 200 OK
Received request for Screenshot_20260813-110054_CPU-Z.jpg | Temp: 0.0 | Thresh: 0.5
INFO:     2a09:bac5:3c74:a8c::10d:25:0 - "POST /api/ocr HTTP/1.1" 200 OK
Received request for Screenshot_20260815-153311_Classroom.jpg | Temp: 0.0 | Thresh: 0.5
INFO:     2a09:bac5:3c74:a8c::10d:25:0 - "POST /api/ocr HTTP/1.1" 200 OK
Received request for Screenshot_20260815-153511_R